## Task 3: Forecast Future Market Trends

### 0. Load Data, Trained Model, and Scaler

We reload the cleaned TSLA data, the trained LSTM model, and the fitted
scaler from Task 2, avoiding the need to retrain. LSTM was selected as the
best-performing model in Task 2 (MAPE of 4.18% vs. ARIMA's 17.24%), so it
is the basis for the future forecast here.

In [6]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import joblib
    from tensorflow.keras.models import load_model

    combined_df = pd.read_csv('../data/processed/combined_assets.csv', parse_dates=['Date'])
    tsla_df = combined_df[combined_df['Ticker'] == 'TSLA'].sort_values('Date').reset_index(drop=True)

    lstm_model = load_model('../data/processed/models/lstm_tsla.keras')
    scaler = joblib.load('../data/processed/models/tsla_scaler.pkl')

    window_size = 60

    print(f"Loaded {len(tsla_df)} rows of TSLA data.")
    print(f"Last date in dataset: {tsla_df['Date'].max()}")
except Exception as e:
    print(f"Error loading data, model, or scaler: {e}")

Loaded 2888 rows of TSLA data.
Last date in dataset: 2026-06-29 00:00:00


d:\Repos\portfolio-optimization\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### 1. Re-establish Train/Test Context

We recreate the same chronological split used in Task 2, so the test-set
residuals (actual vs. predicted error) can be used later to build
confidence intervals around the future forecast.

In [7]:
try:
    train_df = tsla_df[tsla_df['Date'] < '2025-01-01'].reset_index(drop=True)
    test_df = tsla_df[tsla_df['Date'] >= '2025-01-01'].reset_index(drop=True)

    full_close = tsla_df['Adj Close'].values.reshape(-1, 1)
    full_scaled = scaler.transform(full_close)

    print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
except Exception as e:
    print(f"Error re-establishing train/test split: {e}")

Train: 2516 rows | Test: 372 rows


### 2. Compute Historical Residuals

LSTM does not natively produce confidence intervals the way ARIMA does.
To construct realistic uncertainty bounds, we first calculate the model's
prediction residuals (actual − predicted) on the test set. These residuals
capture the model's typical day-to-day error magnitude, which we'll later
sample from to simulate a distribution of plausible future paths.

In [8]:
try:
    train_len = len(train_df)
    test_input_scaled = full_scaled[train_len - window_size:]

    def create_sequences(data, window):
        X, y = [], []
        for i in range(window, len(data)):
            X.append(data[i - window:i, 0])
            y.append(data[i, 0])
        return np.array(X), np.array(y)

    X_test, y_test = create_sequences(test_input_scaled, window_size)
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    test_predictions_scaled = lstm_model.predict(X_test)
    test_predictions = scaler.inverse_transform(test_predictions_scaled).flatten()
    test_actual = test_df['Adj Close'].values[:len(test_predictions)]

    residuals = test_actual - test_predictions
    residual_std = residuals.std()

    print(f"Residual mean: {residuals.mean():.2f}, Residual std: {residual_std:.2f}")
except Exception as e:
    print(f"Error computing residuals: {e}")

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step
Residual mean: -11.07, Residual std: 14.22


### 3. Generate Future Forecasts: Iterative Multi-Step Prediction

Since LSTM predicts one day at a time, forecasting further into the future
requires an iterative approach: predict the next day, append that
prediction to the input sequence, drop the oldest day, and repeat. We
forecast 252 trading days (~12 months) ahead.

To produce confidence intervals, we run this process many times (Monte
Carlo simulation), injecting a small amount of random noise — sampled from
the historical residual distribution — at each step. This simulates the
range of plausible future paths given the model's typical error, and lets
us take percentiles across simulations to form upper/lower bounds.

In [9]:
try:
    def iterative_forecast(model, last_sequence, n_steps, scaler, noise_std=0.0):
        forecast_scaled = []
        current_seq = last_sequence.copy()

        for _ in range(n_steps):
            pred = model.predict(current_seq.reshape(1, window_size, 1), verbose=0)[0, 0]
            if noise_std > 0:
                pred_unscaled = scaler.inverse_transform([[pred]])[0, 0]
                pred_unscaled += np.random.normal(0, noise_std)
                pred = scaler.transform([[pred_unscaled]])[0, 0]
            forecast_scaled.append(pred)
            current_seq = np.append(current_seq[1:], pred)

        forecast_scaled = np.array(forecast_scaled).reshape(-1, 1)
        return scaler.inverse_transform(forecast_scaled).flatten()

    n_future_steps = 252
    last_sequence = full_scaled[-window_size:].flatten()

    point_forecast = iterative_forecast(lstm_model, last_sequence, n_future_steps, scaler, noise_std=0.0)
    print(f"Generated {len(point_forecast)}-day point forecast.")
    print(f"First 5 forecasted prices: {point_forecast[:5]}")
except Exception as e:
    print(f"Error generating iterative forecast: {e}")

Generated 252-day point forecast.
First 5 forecasted prices: [401.06125 412.18463 422.31906 432.22318 441.90387]
